First download the dataset from: https://www.kaggle.com/code/koshirosato/shutterstock-dataset-for-ai-vs-human-gen-image

Extract it into where you want to work

In [1]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

In [2]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.10.0
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Loading Dataset

In [3]:
curr_dir = os.getcwd()
#print(curr_dir)

## Creating Dataset (Only run if data.csv does not exist)

In [13]:
train_csv = pd.read_csv(curr_dir + "\\train.csv")
test_csv = pd.read_csv(curr_dir + "\\test_v2.csv")
test_labels_csv = pd.read_csv(curr_dir + "\\test_v2_labels.csv")

print(f'Train shape: {train_csv.shape}')
print(f'Test shape: {test_csv.shape}')
print(f'Test labels shape: {test_labels_csv}')

Train shape: (79950, 3)
Test shape: (5540, 1)
Test labels shape:                                                      id  label
0     test_data_v2/1a2d9fd3e21b4266aea1f66b30aed157.jpg    1.0
1     test_data_v2/ab5df8f441fe4fbf9dc9c6baae699dc7.jpg    1.0
2     test_data_v2/eb364dd2dfe34feda0e52466b7ce7956.jpg    0.0
3     test_data_v2/f76c2580e9644d85a741a42c6f6b39c0.jpg    0.0
4     test_data_v2/a16495c578b7494683805484ca27cf9f.jpg    0.0
...                                                 ...    ...
5535  test_data_v2/483412064ff74d9d9472d606b65976d9.jpg    1.0
5536  test_data_v2/c0b49ba4081a4197b422dac7c15aea7f.jpg    0.0
5537  test_data_v2/01454aaedec140c0a3ca1f48028c41cf.jpg    0.0
5538  test_data_v2/e9adfea8b67e4791968c4c2bdd8ec343.jpg    1.0
5539  test_data_v2/ba8f4198e8d74d3394fa56c56af23442.jpg    1.0

[5540 rows x 2 columns]


These are CSV files with the filepath + label, will need to use the actual image files for training. Also the image files are not of the same size, though for all of the images one of the dimensions is 768.

The current Train/Test split is 79950 to 5540. I am going to reformat the set so that it forms an 80/20 split for Train/Test. (Validation set will be split from Training set and it will also be 80/20). Final Data Distribution: 64% training, 16% validation, 20% test

In [14]:
test_labels_csv = test_labels_csv[['id', 'label']].rename(columns={'id': 'file_name'})
test_labels_csv['label'] = test_labels_csv['label'].astype('int64')
print(test_labels_csv.head(5))
train_csv = train_csv[['file_name', 'label']]
print(train_csv.head(5))

                                           file_name  label
0  test_data_v2/1a2d9fd3e21b4266aea1f66b30aed157.jpg      1
1  test_data_v2/ab5df8f441fe4fbf9dc9c6baae699dc7.jpg      1
2  test_data_v2/eb364dd2dfe34feda0e52466b7ce7956.jpg      0
3  test_data_v2/f76c2580e9644d85a741a42c6f6b39c0.jpg      0
4  test_data_v2/a16495c578b7494683805484ca27cf9f.jpg      0
                                         file_name  label
0  train_data/a6dcb93f596a43249135678dfcfc17ea.jpg      1
1  train_data/041be3153810433ab146bc97d5af505c.jpg      0
2  train_data/615df26ce9494e5db2f70e57ce7a3a4f.jpg      1
3  train_data/8542fe161d9147be8e835e50c0de39cd.jpg      0
4  train_data/5d81fa12bc3b4cea8c94a6700a477cf2.jpg      1


In [18]:
data = pd.concat([train_csv, test_labels_csv], ignore_index = True)
print(data.sample(frac = 1).head(5))

                                               file_name  label
49631    train_data/65b77453cdc044fd99ac25b189d09f15.jpg      0
28971    train_data/29a2421d72fe43e3a1265582f45d53bb.jpg      0
53085    train_data/aec72301d87746d1b18c93f26125a173.jpg      0
84076  test_data_v2/dbab047ee39f4025a7cd90aa66b40a71.jpg      1
41638    train_data/70e3bb4dcef447239084bfce6a8fc869.jpg      1


## Loading Data

In [4]:
file_name = curr_dir + "\\data.csv"
if not os.path.isfile(file_name):
    data.to_csv(file_name)
else:
    data = pd.read_csv(file_name)

In [5]:
SEED = 694201337

Creating Train, Validation, Test Datasets (note still need to load the images later)

In [7]:
train_df, test_df = train_test_split(data, 
                                     test_size=0.2, 
                                     random_state=SEED, 
                                     stratify=data['label'])
train_df, val_df = train_test_split(train_df,
                                    test_size=0.2,
                                    random_state=SEED,
                                    stratify=train_df['label'])

In [8]:
display(train_df['label'].value_counts())
display(val_df['label'].value_counts())
display(test_df['label'].value_counts())

label
1    27428
0    27285
Name: count, dtype: int64

label
1    6858
0    6821
Name: count, dtype: int64

label
1    8571
0    8527
Name: count, dtype: int64

Now need to convert from file_name in data frames to the actual image. Need CV2 for this